# Part 1 – Foundation Model vs HW3 Model (Olist Reviews)

This notebook compares a pretrained foundation model (`nlptown/bert-base-multilingual-uncased-sentiment`) to my HW3 RandomForest pipeline on 500 Olist review comments. I use the same binary label as HW2/HW3 (review_score ≥ 4 → 1, else 0) and evaluate accuracy, precision, recall, and F1.

In [1]:
!pip install -q transformers scikit-learn pandas numpy

import os
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import pipeline
import joblib

## 1. Load Olist data and sample 500 labeled reviews

We upload the Olist datasets and build a sample of 500 reviews that:
- Have a non-empty `review_comment_message`, and
- Are labeled as positive if `review_score >= 4`, negative otherwise.

In [7]:
from google.colab import files

uploaded = files.upload()  # select the Olist CSVs and model.pkl

os.listdir()

Saving olist_customers_dataset.csv to olist_customers_dataset.csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv


['.config',
 'olist_sellers_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_customers_dataset.csv',
 'olist_products_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'product_category_name_translation.csv',
 'olist_order_payments_dataset.csv',
 'archive.zip',
 'olist_orders_dataset.csv',
 'model.pkl',
 'sample_data']

In [5]:
from google.colab import files

uploaded = files.upload()  # select the Olist CSVs and model.pkl

os.listdir()

Saving model.pkl to model.pkl


['.config', 'archive.zip', 'model.pkl', 'sample_data']

In [10]:
base = "Archive"  # or "" if CSVs are in the root

reviews  = pd.read_csv("olist_order_reviews_dataset.csv")
orders   = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments    = pd.read_csv("olist_order_payments_dataset.csv")
customers   = pd.read_csv("olist_customers_dataset.csv")
products    = pd.read_csv("olist_products_dataset.csv")
sellers     = pd.read_csv("olist_sellers_dataset.csv")

reviews_full = reviews.dropna(subset=["review_comment_message"]).copy()
reviews_full["label"] = (reviews_full["review_score"] >= 4).astype(int)

sample = reviews_full.sample(n=500, random_state=42).reset_index(drop=True)
sample[["order_id", "review_score", "label", "review_comment_message"]].head()

,order_id,review_score,label,review_comment_message
0,301b453872955e08f7aed322d6a0753a,4,1,Um produto como uma carteira somente poderia s...
1,f9281da15642bc69f27307e1dc987b30,5,1,"Entrega no prazo , bom produto ."
2,e747b099854432937d27178c640102bf,5,1,entrega rápida
3,3f3e4083688c07129239450773e84282,5,1,Chegou no prazo e atendeu as minhas espectativas
4,b3b8133868458b6acf08ea6ffc22e3f2,5,1,Logística ótima entregue antes do prazo previs...


## 2. Foundation model: nlptown BERT on review comments

We use the HuggingFace `nlptown/bert-base-multilingual-uncased-sentiment` model to score the 500 review comments. The model outputs star ratings (1–5); we map these back to the same binary label (≥ 4 stars → 1, else 0) and compute classification metrics.

In [11]:
sentiment = pipeline(
    task="sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

texts = sample["review_comment_message"].tolist()
foundation_preds = sentiment(texts, truncation=True)

def label_to_binary(label_str: str) -> int:
    stars = int(label_str.split()[0])  # "4 stars" -> 4
    return 1 if stars >= 4 else 0

foundation_binary = [label_to_binary(r["label"]) for r in foundation_preds]
sample["foundation_pred"] = foundation_binary

y_true = sample["label"].values
y_pred_foundation = sample["foundation_pred"].values

metrics_foundation = {
    "accuracy": accuracy_score(y_true, y_pred_foundation),
    "precision": precision_score(y_true, y_pred_foundation, zero_division=0),
    "recall": recall_score(y_true, y_pred_foundation, zero_division=0),
    "f1": f1_score(y_true, y_pred_foundation, zero_division=0),
}
metrics_foundation

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

{'accuracy': 0.834,
 'precision': 0.951310861423221,
 'recall': 0.7839506172839507,
 'f1': 0.8595600676818951}

In [12]:
sample_order_ids = sample["order_id"].unique()

orders_sub = orders[orders["order_id"].isin(sample_order_ids)].copy()
order_items_sub = order_items[order_items["order_id"].isin(sample_order_ids)].copy()
payments_sub = payments[payments["order_id"].isin(sample_order_ids)].copy()

# timestamps
orders_sub["order_purchase_timestamp"] = pd.to_datetime(orders_sub["order_purchase_timestamp"])
orders_sub["order_delivered_customer_date"] = pd.to_datetime(orders_sub["order_delivered_customer_date"])
orders_sub["order_estimated_delivery_date"] = pd.to_datetime(orders_sub["order_estimated_delivery_date"])

orders_sub["delivery_days"] = (
    orders_sub["order_delivered_customer_date"] - orders_sub["order_purchase_timestamp"]
).dt.days

orders_sub["delivery_vs_estimated"] = (
    orders_sub["order_delivered_customer_date"] - orders_sub["order_estimated_delivery_date"]
).dt.days

orders_sub["order_purchase_dow"] = orders_sub["order_purchase_timestamp"].dt.dayofweek

items_agg = (
    order_items_sub
    .groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        n_items=("order_item_id", "count"),
        n_sellers=("seller_id", "nunique"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

payments_agg = (
    payments_sub
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type=("payment_type", lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan),
    )
    .reset_index()
)

features = (
    orders_sub
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(
        order_items_sub[["order_id", "product_id", "seller_id"]].drop_duplicates("order_id"),
        on="order_id", how="left"
    )
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
)

features = features.rename(columns={"product_category_name": "product_category"})

feature_cols = [
    "delivery_days",
    "delivery_vs_estimated",
    "order_purchase_dow",
    "total_price",
    "total_freight",
    "n_items",
    "n_sellers",
    "avg_price",
    "payment_value",
    "payment_installments",
    "product_category",
    "seller_state",
    "payment_type",
]

X_hw3 = features.set_index("order_id")[feature_cols]
X_hw3.head()

,delivery_days,delivery_vs_estimated,order_purchase_dow,total_price,total_freight,n_items,n_sellers,avg_price,payment_value,payment_installments,product_category,seller_state,payment_type
order_id,,,,,,,,,,,,,
01855f880aae9a984c7c33b26fcf2e02,5.0,-7.0,3,78.98,16.54,2.0,1.0,39.49,95.52,2,beleza_saude,SP,credit_card
d83706c29baf36eedf5e8adfb0da304e,14.0,-16.0,0,27.90,16.05,1.0,1.0,27.90,43.95,1,telefonia,SP,boleto
6c12feac9a308e1382d9b19cca7f20b2,4.0,-16.0,4,208.90,12.07,1.0,1.0,208.90,220.97,4,beleza_saude,MG,credit_card
689624d719c9fffa2c4ff8f71da712e4,11.0,-18.0,2,69.00,22.19,1.0,1.0,69.00,91.19,2,consoles_games,PR,credit_card
53f3f52eca823461e5ee857dd46fb101,19.0,-6.0,6,278.00,16.70,1.0,1.0,278.00,294.70,8,relogios_presentes,SP,credit_card


## 3. HW3 RandomForest pipeline on the same 500 orders

Next, we reconstruct the HW3 feature set for the same 500 orders:
- Delivery features: `delivery_days`, `delivery_vs_estimated`, `order_purchase_dow`
- Order aggregates: `total_price`, `total_freight`, `n_items`, `n_sellers`, `avg_price`
- Payment features: `payment_value`, `payment_installments`, `payment_type`
- Categorical: `product_category`, `seller_state`

We then train a RandomForest inside a `ColumnTransformer + Pipeline` and evaluate on this sample.

In [24]:
# Drop rows with any NaN in feature columns
mask = ~X_hw3_common.isna().any(axis=1)

X_hw3_clean = X_hw3_common[mask].copy()
y_clean = y_common[mask]

print("Original rows:", X_hw3_common.shape[0])
print("Rows after dropping NaNs:", X_hw3_clean.shape[0])

Original rows: 500
Rows after dropping NaNs: 480


In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    "delivery_days",
    "delivery_vs_estimated",
    "order_purchase_dow",
    "total_price",
    "total_freight",
    "n_items",
    "n_sellers",
    "avg_price",
    "payment_value",
    "payment_installments",
]

categorical_features = [
    "product_category",
    "seller_state",
    "payment_type",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)

hw3_pipeline_colab = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", rf),
    ]
)

In [26]:
hw3_pipeline_colab.fit(X_hw3_clean, y_clean)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['delivery_days',
                                                   'delivery_vs_estimated',
                                                   'order_purchase_dow',
                                                   'total_price',
                                                   'total_freight', 'n_items',
                                                   'n_sellers', 'avg_price',
                                                   'payment_value',
                                                   'payment_installments']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_category',
                                                   'seller_state',
                                                   'payment_type'])])),
                ('model',
                 RandomForestClassifier(max_depth=10, n_jobs=-1,
                                        random_state=42))])

In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_hw3 = hw3_pipeline_colab.predict(X_hw3_clean)

metrics_hw3 = {
    "accuracy": accuracy_score(y_clean, y_pred_hw3),
    "precision": precision_score(y_clean, y_pred_hw3, zero_division=0),
    "recall": recall_score(y_clean, y_pred_hw3, zero_division=0),
    "f1": f1_score(y_clean, y_pred_hw3, zero_division=0),
}
metrics_hw3

{'accuracy': 0.84375,
 'precision': 0.8110831234256927,
 'recall': 1.0,
 'f1': 0.8956884561891516}

## 4. Metric comparison

The table below compares accuracy, precision, recall, and F1 for both models on the same 500 examples.

In [23]:
comparison = pd.DataFrame([
    {"model": "Foundation (nlptown BERT)", **metrics_foundation},
    {"model": "HW3 RandomForest pipeline", **metrics_hw3},
])

comparison

,model,accuracy,precision,recall,f1
0,Foundation (nlptown BERT),0.83400,0.951311,0.783951,0.859560
1,HW3 RandomForest pipeline,0.84375,0.811083,1.000000,0.895688


## 5. Reflection

On this 500-review sample, the foundation model and my HW3 RandomForest pipeline perform similarly overall but with different trade-offs. The foundation model (`nlptown/bert-base-multilingual-uncased-sentiment`) reaches accuracy ≈ 0.83 and F1 ≈ 0.86. It is very precise on positives (precision ≈ 0.95) but misses more true positives (recall ≈ 0.78). In contrast, the HW3 RandomForest achieves slightly higher accuracy (≈ 0.84) and a stronger F1 (≈ 0.90) by capturing all positives in this sample (recall = 1.0) at the cost of a lower precision (≈ 0.81). In other words, the foundation model is more conservative, while the HW3 model is more “inclusive” and will flag more borderline positive cases.

From a modeling perspective, the foundation model is attractive because it requires **no task-specific feature engineering or labels beyond the review text**; I only needed raw comments. It also generalizes naturally to other languages and domains where we may not have structured order data. However, it is relatively heavy: loading the model and running 500 predictions is slower and more resource-intensive than scoring with a compact scikit-learn pipeline. The HW3 model, by contrast, depends on a carefully engineered feature set (delivery times, price aggregates, payments, etc.) and a labeled training set, but once those exist it is fast, cheap to run, and easy to inspect or retrain.

For a real production system, I would probably start with the **HW3 RandomForest** for this particular business problem. It is already wired into an MLOps-friendly pipeline, has transparent input features that operations and business teams understand, and performs slightly better on F1 in this sample. The foundation model is a great second opinion and could be used as an ensemble or fallback when text is available but structured features are missing. Over time, combining both (for example, using the foundation model’s embeddings as additional features) could yield the best of both worlds: robust, general language understanding plus business-specific signals from the Olist data.
